# Q1/Q2 measurement and cohort checks

**Question:** Are measurement concepts/units, exposure provenance, event timing and covariate availability consistent with the analysis?

**Inputs:** Authorized baseline/follow-up/long measurement inputs and previously generated Q1/Q2 model summaries.

**Outputs:** Measurement/unit audits, follow-up checks, CRP 30-/90-day sensitivity, BMI/eGFR checks and missingness summaries.

Run after the main Q1/Q2 outputs are available. The broader baseline cohort and complete-case Cox cohort have distinct denominators. Optional inputs remain explicitly optional; unavailable evidence is not converted into a successful check. Historical review-comment tracking tables and duplicate manuscript model/table refits were removed.


## Locate required inputs


In [ ]:
from pathlib import Path
import math
import re
import warnings

import numpy as np
import pandas as pd

try:
    from scipy import stats
except Exception:
    stats = None

try:
    import statsmodels.formula.api as smf
except Exception:
    smf = None

try:
    from lifelines import CoxPHFitter
except Exception:
    CoxPHFitter = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

import os
DATA_DIR = Path(os.environ["CAPSTONE_DATA_DIR"]).expanduser()
RUN_TABLE_DIR = Path(os.environ["CAPSTONE_OUTPUT_DIR"]).expanduser() / "tables"
OUT_DIR = Path(os.environ["CAPSTONE_OUTPUT_DIR"]).expanduser() / "quality_checks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_FILES_INDEX = sorted(set(DATA_DIR.rglob("*.csv")) | set(RUN_TABLE_DIR.rglob("*.csv")))

def find_data_file(filename, preferred_parts=()):
    current = RUN_TABLE_DIR / filename
    if current.is_file():
        return current
    matches = [p for p in DATA_FILES_INDEX if p.name == filename]
    if not matches:
        return DATA_DIR / filename
    preferred = [part.lower() for part in preferred_parts]
    def score(path):
        s = str(path).lower()
        return sum(part in s for part in preferred)
    return sorted(matches, key=lambda p: (-score(p), len(str(p)), str(p)))[0]

Q1_FILES = {
    'q1_baseline': find_data_file('q1_covariate_table_v3_bmi.csv', ['q1']),
    'q1_cox': find_data_file('analysis_base_clean.csv', ['q1']),
    'q1_followup': find_data_file('q1_competing_risk_data.csv', ['q1']),
    'q1_vitd_long': find_data_file('vitd_long_clean.csv', ['q1']),
    'q1_vitd_baseline_versions': find_data_file('vitd_baseline_versions.csv', ['q1']),
    'q1_vitd_unit_audit': find_data_file('audit_vitd_units.csv', ['q1']),
    'q1_crp_unit_audit': find_data_file('audit_crp_units.csv', ['q1', 'crp']),
    'q1_crp_unit_map': find_data_file('audit_crp_unit_map.csv', ['q1', 'crp']),
    'q1_primary_cox_table': find_data_file('final_table2_primary_cox.csv', ['q1']),
    'q1_finegray': find_data_file('q1_finegray_shr_rcheck.csv', ['q1']),
    'q1_bmi_sensitivity': find_data_file('q1_bmi_sensitivity_summary.csv', ['q1']),
    'q1_bmi_audit': find_data_file('q1_bmi_audit.csv', ['q1', 'bmi']),
    'q1_bmi_timing_audit': find_data_file('q1_bmi_timing_audit_rcheck.csv', ['q1', 'bmi']),
    'q1_recovered_bmi_timing_audit': find_data_file('q1_recovered_bmi_timing_audit.csv', ['q1', 'bmi']),
    'q1_egfr_baseline': find_data_file('q1_egfr_baseline.csv', ['q1', 'egfr']),
    'q1_concept_sensitivity': find_data_file('q1_conceptset_sensitivity.csv', ['q1']),
    'q1_formal_concepts': find_data_file('formal_concept_sets.csv', ['q1']),
    'q1_finalized_concepts': find_data_file('finalized_concept_sets.csv', ['q1']),
    'q1_subgroups': find_data_file('q1_subgroups_all.csv', ['q1']),
    'q1_interactions': find_data_file('q1_interaction_tests.csv', ['q1']),
}

Q2_FILES = {
    'q2_loglog': find_data_file('q2_loglog_results.csv', ['q2', 'results']),
    'q2_models': find_data_file('q2_extended_model_loglog.csv', ['q2', 'results']),
    'q2_full_coefficients': find_data_file('q2_extended_model_full_coefficients.csv', ['q2', 'results']),
    'q2_dataset': find_data_file('q2_extended_model_dataset.csv', ['q2', 'results']),
    'q2_coverage': find_data_file('q2_covariate_coverage.csv', ['q2', 'results']),
    'q2_egfr_baseline': find_data_file('q2_egfr_baseline.csv', ['q2', 'egfr']),
}

Q2_MODEL_LABELS = {
    'A_core_current': 'Model A: age, sex, winter season, BMI',
    'B_extended_race_diabetes_SES': 'Model B: Model A + race/ethnicity, diabetes, SES',
    'C_extended_plus_vitd_supplement': 'Model C: Model B + recorded vitamin D supplement use',
}

if DATA_FILES_INDEX:
    print('\nCSV files indexed under DATA_DIR:')
    for path in DATA_FILES_INDEX:
        try:
            display_path = path.relative_to(DATA_DIR)
        except ValueError:
            display_path = path
        print(f'  {display_path}')
for key, path in Q1_FILES.items():
    print(f'  {key}: {path}')
for key, path in Q2_FILES.items():
    print(f'  {key}: {path}')


## Summary table helpers


In [ ]:
TABLES = {}
MANIFEST = []

def read_csv_required(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}')
    return pd.read_csv(path, **kwargs)

def read_csv_optional(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f'Optional file not found: {path}')
        return None
    return pd.read_csv(path, **kwargs)

def fmt_int(x):
    if pd.isna(x):
        return ''
    return f'{int(round(float(x))):,}'

def fmt_num(x, digits=1):
    if pd.isna(x):
        return ''
    return f'{float(x):.{digits}f}'

def fmt_p(x):
    if pd.isna(x):
        return ''
    x = float(x)
    if x < 0.001:
        return '<0.001'
    return f'{x:.3f}'

def fmt_percent(x, digits=1):
    if pd.isna(x):
        return ''
    return f'{100 * float(x):.{digits}f}'

def n_pct(n, denom):
    if denom == 0 or pd.isna(denom):
        return ''
    return f'{int(n):,} ({100 * n / denom:.1f})'

def median_iqr(values, digits=1):
    x = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if x.empty:
        return ''
    q1, med, q3 = x.quantile([0.25, 0.50, 0.75])
    return f'{med:.{digits}f} ({q1:.{digits}f}-{q3:.{digits}f})'

def range_cell(values, digits=1):
    x = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if x.empty:
        return ''
    return f'{x.min():.{digits}f}-{x.max():.{digits}f}'

def missing_cell(series):
    s = pd.Series(series)
    return n_pct(int(s.isna().sum()), len(s))

def clean_sheet_name(name):
    name = re.sub(r'[\\/*?:\[\]]', '_', name)
    return name[:31]

def df_to_markdown(df):
    cols = list(df.columns)
    lines = []
    lines.append('| ' + ' | '.join(map(str, cols)) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in df.iterrows():
        vals = [str(row[c]).replace('\n', '<br>').replace('|', '\\|') for c in cols]
        lines.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join(lines) + '\n'

def save_table(name, df, title=None, png_preview=True):
    df = df.copy()
    TABLES[name] = df
    csv_path = OUT_DIR / f'{name}.csv'
    md_path = OUT_DIR / f'{name}.md'
    html_path = OUT_DIR / f'{name}.html'
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    md_title = f'# {title or name}\n\n' if title else ''
    md_path.write_text(md_title + df_to_markdown(df), encoding='utf-8')
    html_title = f'<h3>{title}</h3>' if title else ''
    html_path.write_text(html_title + df.to_html(index=False, escape=False), encoding='utf-8')
    png_path = None
    if png_preview and plt is not None and len(df) <= 80:
        png_path = OUT_DIR / f'{name}.png'
        fig_w = min(max(8, 1.5 * len(df.columns)), 18)
        fig_h = min(max(2.5, 0.35 * len(df) + 1.5), 30)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        ax.axis('off')
        if title:
            ax.set_title(title, fontsize=11, loc='left', pad=8)
        tbl = ax.table(cellText=df.astype(str).values, colLabels=df.columns, loc='center', cellLoc='left')
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(7)
        tbl.scale(1, 1.25)
        fig.tight_layout()
        fig.savefig(png_path, dpi=220, bbox_inches='tight')
        plt.close(fig)
    MANIFEST.append({'name': name, 'rows': len(df), 'columns': len(df.columns), 'csv': str(csv_path), 'markdown': str(md_path), 'html': str(html_path), 'png_preview': str(png_path) if png_path else ''})
    print(f'Saved {name}: {len(df)} rows')
    return df

def write_excel_workbook(filename='q1_q2_manuscript_tables_latest.xlsx'):
    try:
        import openpyxl  # noqa: F401
    except Exception:
        print('openpyxl is not installed; skipping Excel workbook export. CSV/Markdown/HTML outputs remain available.')
        return None
    xlsx_path = OUT_DIR / filename
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
        used = set()
        for name, df in TABLES.items():
            sheet = clean_sheet_name(name)
            base = sheet
            i = 1
            while sheet in used:
                suffix = f'_{i}'
                sheet = (base[:31-len(suffix)] + suffix)
                i += 1
            used.add(sheet)
            df.to_excel(writer, sheet_name=sheet, index=False)
    print('Saved Excel workbook:', xlsx_path)
    return xlsx_path


def write_markdown_workbook(filename='q1_q2_manuscript_tables_latest.md'):
    md_path = OUT_DIR / filename
    lines = [
        '# Q1/Q2 Manuscript Tables - Latest Server Outputs',
        '',
        'This file aggregates the table outputs generated by this notebook in manuscript order. Q3/VDR tables are excluded.',
        '',
        f'Output directory: `{OUT_DIR}`',
        '',
        '## Table Index',
        '',
        '| Order | Table key | Rows | Columns | CSV | Individual MD |',
        '| ---: | --- | ---: | ---: | --- | --- |',
    ]
    for i, item in enumerate(MANIFEST, 1):
        lines.append(f"| {i} | `{item['name']}` | {item['rows']} | {item['columns']} | `{Path(item['csv']).name}` | `{Path(item['markdown']).name}` |")
    lines.append('')
    for i, (name, df) in enumerate(TABLES.items(), 1):
        lines.append(f'## {i}. {name}')
        lines.append('')
        match = next((m for m in MANIFEST if m['name'] == name), None)
        if match:
            lines.append(f"Source files: `{Path(match['csv']).name}`, `{Path(match['markdown']).name}`, `{Path(match['html']).name}`")
            lines.append('')
        lines.append(df_to_markdown(df))
        lines.append('')
    md_path.write_text('\n'.join(lines), encoding='utf-8')
    print('Saved Markdown workbook:', md_path)
    return md_path


## Read and validate inputs


In [ ]:
q1_baseline = read_csv_required(Q1_FILES['q1_baseline'])
q1_cox = read_csv_required(Q1_FILES['q1_cox'])
q1_followup = read_csv_required(Q1_FILES['q1_followup'])
q1_vitd_long = read_csv_required(Q1_FILES['q1_vitd_long'])

q2_dataset = read_csv_required(Q2_FILES['q2_dataset'])
q2_loglog = read_csv_required(Q2_FILES['q2_loglog'])
q2_models = read_csv_required(Q2_FILES['q2_models'])
q2_full = read_csv_optional(Q2_FILES['q2_full_coefficients'])
q2_coverage = read_csv_optional(Q2_FILES['q2_coverage'])

validation_rows = []
for label, df, key in [
    ('Q1 baseline covariates', q1_baseline, 'person_id'),
    ('Q1 Cox modeling cohort', q1_cox, 'person_id'),
    ('Q1 follow-up/competing-risk cohort', q1_followup, 'person_id'),
    ('Q1 clean long 25(OH)D records', q1_vitd_long, None),
    ('Q2 paired/model dataset', q2_dataset, 'person_id'),
]:
    duplicate_keys = ''
    if key is not None and key in df.columns:
        duplicate_keys = int(df[key].duplicated().sum())
    validation_rows.append({'source': label, 'rows': len(df), 'columns': len(df.columns), 'duplicate_person_id': duplicate_keys})

source_validation = pd.DataFrame(validation_rows)
save_table('00_source_validation_q1_q2', source_validation, 'Source validation for Q1/Q2 table redraw', png_preview=False)

resolved_rows = []
for family, mapping in [('Q1', Q1_FILES), ('Q2', Q2_FILES)]:
    for key, path in mapping.items():
        path = Path(path)
        resolved_rows.append({
            'Family': family,
            'Key': key,
            'Expected filename': path.name,
            'Resolved path': str(path),
            'Exists': path.exists(),
            'Required for core tables': key not in {'q1_primary_cox_table', 'q1_subgroups', 'q1_interactions', 'q2_full_coefficients'},
        })
resolved_inputs = pd.DataFrame(resolved_rows)
save_table('00_resolved_input_files_q1_q2', resolved_inputs, 'Resolved input files under server data folder', png_preview=False)


## Vitamin D concept and unit checks


In [ ]:
vitd_unit_audit = read_csv_required(Q1_FILES['q1_vitd_unit_audit'])
included_vitd_cids = {3020149, 40765040, 3049536}
vitd_long_included = q1_vitd_long[q1_vitd_long['cid'].isin(included_vitd_cids)].copy()

concept_label = {
    3020149: '25(OH)D3',
    40765040: 'Total 25(OH)D from D3 plus D2',
    3049536: '25(OH)D2',
}
concept_counts = (
    vitd_long_included.groupby('cid')
    .agg(Records=('vitd_ng_ml', 'size'), Persons=('person_id', 'nunique'))
    .reset_index()
)
concept_counts['Concept'] = concept_counts['cid'].map(concept_label)
concept_counts = concept_counts[['Concept', 'cid', 'Records', 'Persons']].rename(columns={'cid': 'OMOP concept ID'})
concept_counts['Records'] = concept_counts['Records'].map(fmt_int)
concept_counts['Persons'] = concept_counts['Persons'].map(fmt_int)
final_row = pd.DataFrame([{'Concept': 'Final cleaned records', 'OMOP concept ID': 'NA', 'Records': fmt_int(len(vitd_long_included)), 'Persons': fmt_int(vitd_long_included['person_id'].nunique())}])
concept_counts = pd.concat([concept_counts, final_row], ignore_index=True)
save_table('q1_supp_vitd_clean_concept_counts', concept_counts, 'Supplementary table. Final clean 25(OH)D concept counts')

def classify_unit_issue(unit_name):
    u = str(unit_name).lower()
    if 'nanogram per milliliter' in u:
        return 'Recorded ng/mL'
    if 'microgram per liter' in u:
        return 'Microgram per L equivalent'
    if 'picogram' in u or 'pg/ml' in u:
        return 'pg/picogram labels with ng/mL-scale distributions'
    if u in ['nan', 'null', 'no matching concept', 'no value', 'none', '']:
        return 'Missing/no matching/no value unit labels'
    return 'Other rare unit labels reviewed during harmonization'

unit_detail = vitd_unit_audit[vitd_unit_audit['cid'].isin(included_vitd_cids)].copy()
unit_detail['Harmonization issue'] = unit_detail['unit_name'].map(classify_unit_issue)
unit_detail['Concept'] = unit_detail['cid'].map(concept_label)
unit_detail = unit_detail.rename(columns={'cid': 'OMOP concept ID', 'test_name': 'Concept name', 'unit_id': 'Unit concept ID', 'unit_name': 'Unit label', 'n_rows': 'Records', 'n_people': 'Persons', 'p5': 'P5', 'median_val': 'Median', 'p95': 'P95'})
unit_detail = unit_detail[['Concept', 'OMOP concept ID', 'Concept name', 'Unit concept ID', 'Unit label', 'Harmonization issue', 'Records', 'Persons', 'P5', 'Median', 'P95']]
save_table('q1_supp_vitd_unit_harmonization_detail', unit_detail, 'Supplementary table. 25(OH)D unit harmonization detail', png_preview=False)

unit_summary = (
    unit_detail.groupby('Harmonization issue', dropna=False)
    .agg(Records=('Records', 'sum'), Person_unit_count=('Persons', 'sum'), Unit_labels=('Unit label', lambda x: '; '.join(sorted(set(map(str, x))))))
    .reset_index()
)
unit_summary['Records'] = unit_summary['Records'].map(fmt_int)
unit_summary['Person-unit count'] = unit_summary['Person_unit_count'].map(fmt_int)
unit_summary = unit_summary.drop(columns=['Person_unit_count'])
save_table('q1_supp_vitd_unit_harmonization_summary', unit_summary, 'Supplementary table. 25(OH)D unit harmonization summary')


## Exposure provenance


In [ ]:
# 25(OH)D concept-source, baseline selection, events, and person-years audit
vitd_versions = read_csv_optional(Q1_FILES['q1_vitd_baseline_versions'])
concept_name_map = {
    3020149: '25(OH)D3',
    40765040: 'Total 25(OH)D from D3 plus D2',
    3049536: '25(OH)D2',
}

clean_concept = q1_vitd_long.copy()
clean_concept['cid_numeric'] = pd.to_numeric(clean_concept['cid'], errors='coerce')
clean_summary = (
    clean_concept[clean_concept['cid_numeric'].isin(concept_name_map)]
    .groupby('cid_numeric')
    .agg(clean_records=('vitd_ng_ml', 'size'), clean_persons=('person_id', 'nunique'))
    .reset_index()
    .rename(columns={'cid_numeric': 'OMOP concept ID'})
)
clean_summary['Concept'] = clean_summary['OMOP concept ID'].map(concept_name_map)

if vitd_versions is not None:
    source_col = 'src_main' if 'src_main' in vitd_versions.columns else 'src_old' if 'src_old' in vitd_versions.columns else None
else:
    source_col = None

if source_col is not None:
    src = vitd_versions[['person_id', source_col]].copy()
    src[source_col] = pd.to_numeric(src[source_col], errors='coerce')

    primary_source = (
        q1_baseline[['person_id', 'vitd_ng_ml']]
        .merge(src, on='person_id', how='left')
        .merge(q1_followup[['person_id', 'event_mi_or_stroke', 'follow_up_years']], on='person_id', how='left')
    )
    cox_source = (
        q1_cox[['person_id', 'vitd_main', 'event_mi_or_stroke', 'follow_up_years']]
        .merge(src, on='person_id', how='left')
    )

    def summarize_source(df, cohort_label):
        rows = []
        for cid, sub in df.dropna(subset=[source_col]).groupby(source_col):
            cid_int = int(cid)
            py = pd.to_numeric(sub.get('follow_up_years'), errors='coerce').sum() if 'follow_up_years' in sub.columns else np.nan
            events = pd.to_numeric(sub.get('event_mi_or_stroke'), errors='coerce').fillna(0).sum() if 'event_mi_or_stroke' in sub.columns else np.nan
            rows.append({
                'Cohort': cohort_label,
                'OMOP concept ID': cid_int,
                'Concept': concept_name_map.get(cid_int, str(cid_int)),
                'Baseline-selected persons': int(sub['person_id'].nunique()),
                'Events': int(events) if pd.notna(events) else np.nan,
                'Person-years': py,
            })
        return pd.DataFrame(rows)

    selected_summary = pd.concat([
        summarize_source(primary_source, 'Primary analytic cohort'),
        summarize_source(cox_source, 'Cox modeling cohort'),
    ], ignore_index=True)
    concept_event = selected_summary.merge(clean_summary, on=['OMOP concept ID', 'Concept'], how='outer')
    concept_event['Clean records'] = concept_event['clean_records'].map(fmt_int)
    concept_event['Clean persons'] = concept_event['clean_persons'].map(fmt_int)
    concept_event['Baseline-selected persons'] = concept_event['Baseline-selected persons'].map(fmt_int)
    concept_event['Events'] = concept_event['Events'].map(fmt_int)
    concept_event['Person-years'] = concept_event['Person-years'].map(fmt_int)
    concept_event = concept_event[['Cohort', 'Concept', 'OMOP concept ID', 'Clean records', 'Clean persons', 'Baseline-selected persons', 'Events', 'Person-years']]
else:
    concept_event = pd.DataFrame([{
        'Cohort': 'Not generated',
        'Concept': '',
        'OMOP concept ID': '',
        'Clean records': '',
        'Clean persons': '',
        'Baseline-selected persons': '',
        'Events': '',
        'Person-years': '',
        'Reason': 'vitd_baseline_versions.csv with src_old/src_main was not found under DATA_DIR.',
    }])

save_table('q1_supp_vitd_concept_source_events_c115', concept_event, 'Supplementary table. 25(OH)D concept-source, baseline selection, events, and person-years', png_preview=False)


## Repeated-measure coverage


In [ ]:
def repeated_measure_audit(cohort_person_ids, cohort_label, baseline_df):
    ids = pd.Series(cohort_person_ids).dropna().unique()
    long = q1_vitd_long[q1_vitd_long['person_id'].isin(ids)].copy()
    baseline = baseline_df[baseline_df['person_id'].isin(ids)].copy()
    baseline['index_day'] = pd.to_datetime(baseline['index_date'], errors='coerce').dt.date
    long['day_as_date'] = pd.to_datetime(long['day'], errors='coerce').dt.date

    rec_per_person = long.groupby('person_id').size()
    day_counts = long.groupby(['person_id', 'day_as_date']).size().reset_index(name='records_on_day')
    any_same_day_persons = day_counts.loc[day_counts['records_on_day'] > 1, 'person_id'].nunique()
    baseline_day = long.merge(baseline[['person_id', 'index_day']], left_on=['person_id', 'day_as_date'], right_on=['person_id', 'index_day'], how='inner')
    baseline_day_counts = baseline_day.groupby('person_id').size()
    baseline_multi_persons = int((baseline_day_counts > 1).sum())
    baseline_multi_records = int(baseline_day_counts.loc[baseline_day_counts > 1].sum()) if baseline_multi_persons > 0 else 0

    return pd.DataFrame([
        {'Cohort': cohort_label, 'Metric': 'Participants', 'Value': fmt_int(len(ids))},
        {'Cohort': cohort_label, 'Metric': 'Eligible clean 25(OH)D records', 'Value': fmt_int(len(long))},
        {'Cohort': cohort_label, 'Metric': 'Records per participant, median (IQR)', 'Value': median_iqr(rec_per_person, 0)},
        {'Cohort': cohort_label, 'Metric': 'Records per participant, maximum', 'Value': fmt_int(rec_per_person.max())},
        {'Cohort': cohort_label, 'Metric': 'Participants with >1 eligible record', 'Value': fmt_int((rec_per_person > 1).sum())},
        {'Cohort': cohort_label, 'Metric': 'Participants with any same-day multiple eligible records', 'Value': fmt_int(any_same_day_persons)},
        {'Cohort': cohort_label, 'Metric': 'Participants with multiple eligible records on baseline date', 'Value': fmt_int(baseline_multi_persons)},
        {'Cohort': cohort_label, 'Metric': 'Baseline-date records among those participants', 'Value': fmt_int(baseline_multi_records)},
        {'Cohort': cohort_label, 'Metric': 'Baseline 25(OH)D range, ng/mL', 'Value': range_cell(baseline['vitd_ng_ml'] if 'vitd_ng_ml' in baseline.columns else baseline['vitd_main'], 1)},
        {'Cohort': cohort_label, 'Metric': 'Baseline 25(OH)D median (IQR), ng/mL', 'Value': median_iqr(baseline['vitd_ng_ml'] if 'vitd_ng_ml' in baseline.columns else baseline['vitd_main'], 1)},
    ])

repeat_primary = repeated_measure_audit(q1_baseline['person_id'], 'Primary analytic cohort', q1_baseline)
cox_baseline_proxy = q1_baseline[q1_baseline['person_id'].isin(q1_cox['person_id'])].copy()
repeat_cox = repeated_measure_audit(q1_cox['person_id'], 'Cox modeling cohort', cox_baseline_proxy)
repeat_audit = pd.concat([repeat_primary, repeat_cox], ignore_index=True)
save_table('q1_supp_vitd_repeated_measure_audit', repeat_audit, 'Supplementary table. Repeated 25(OH)D measurement audit')


## Outcome concept definitions


In [ ]:
concept_source = read_csv_optional(Q1_FILES['q1_formal_concepts'])
if concept_source is None:
    concept_source = read_csv_optional(Q1_FILES['q1_finalized_concepts'])

if concept_source is not None:
    concept_table = concept_source.copy()
    rename_map = {
        'condition_concept_id': 'OMOP concept ID',
        'concept_id': 'OMOP concept ID',
        'concept_name': 'Concept name',
        'category': 'Concept set category',
        'n_records': 'Records',
        'n_people': 'Persons',
    }
    concept_table = concept_table.rename(columns={k: v for k, v in rename_map.items() if k in concept_table.columns})
    preferred = [c for c in ['Concept set category', 'OMOP concept ID', 'Concept name', 'Records', 'Persons'] if c in concept_table.columns]
    concept_table = concept_table[preferred + [c for c in concept_table.columns if c not in preferred]]
    save_table('q1_supp_omop_concept_set_reproducibility', concept_table, 'Supplementary Table 1. OMOP concept sets for baseline exclusion and incident MI/stroke ascertainment', png_preview=False)


## Follow-up and event timing


In [ ]:
# follow-up and event-timing audit
SECONDS_PER_DAY = 24 * 60 * 60
DAYS_PER_YEAR = 365.25

def event_timing_table(cohort_ids, label):
    df = q1_followup[q1_followup['person_id'].isin(pd.Series(cohort_ids).dropna().unique())].copy()
    py = pd.to_numeric(df['follow_up_years'], errors='coerce').sum()
    events = pd.to_numeric(df['event_mi_or_stroke'], errors='coerce').fillna(0).astype(int)
    event_df = df.loc[events == 1].copy()
    event_years = pd.to_numeric(event_df['follow_up_years'], errors='coerce')
    event_days = pd.to_numeric(event_df['follow_up_days'], errors='coerce') if 'follow_up_days' in event_df.columns else event_years * DAYS_PER_YEAR
    return pd.DataFrame([{
        'Cohort': label,
        'N': fmt_int(len(df)),
        'Events': fmt_int(events.sum()),
        'Person-years': fmt_int(py),
        'Event rate per 1,000 PY': fmt_num(events.sum() / py * 1000, 2) if py > 0 else '',
        'Follow-up, years, median (IQR)': median_iqr(df['follow_up_years'], 2),
        'Follow-up, years, range': range_cell(df['follow_up_years'], 2),
        'Time from index to event, years, median (IQR)': median_iqr(event_years, 2),
        'Time from index to event, years, range': range_cell(event_years, 3),
        'Events within 30 days': fmt_int((event_days <= 30).sum()),
        'Events within 90 days': fmt_int((event_days <= 90).sum()),
        'Events within 1 year': fmt_int((event_days <= 365).sum()),
    }])

event_timing = pd.concat([
    event_timing_table(q1_followup['person_id'], 'Primary analytic/follow-up cohort'),
    event_timing_table(q1_cox['person_id'], 'Cox modeling cohort'),
], ignore_index=True)
save_table('q1_supp_followup_event_timing_c124_c153', event_timing, 'Supplementary table. Follow-up and event timing audit')


## CRP units, timing and window sensitivity


In [ ]:
# CRP unit, timing distribution, and 30-/90-day timing sensitivity
crp_unit_audit = read_csv_optional(Q1_FILES['q1_crp_unit_audit'])
crp_unit_map = read_csv_optional(Q1_FILES['q1_crp_unit_map'])

if crp_unit_audit is not None:
    save_table('q2_supp_crp_unit_audit_detail_c126', crp_unit_audit, 'Supplementary table. CRP unit audit detail', png_preview=False)
if crp_unit_map is not None:
    save_table('q2_supp_crp_unit_mapping_c126', crp_unit_map, 'Supplementary table. CRP unit mapping decisions', png_preview=False)

q2_timing = q2_dataset.copy()
q2_timing['days_before_numeric'] = pd.to_numeric(q2_timing.get('days_before'), errors='coerce')
q2_timing['crp_numeric'] = pd.to_numeric(q2_timing.get('crp_mgL'), errors='coerce')
q2_timing['ambiguous_numeric'] = pd.to_numeric(q2_timing.get('ambiguous'), errors='coerce') if 'ambiguous' in q2_timing.columns else np.nan

timing_bins = [-np.inf, -1, 0, 30, 90, 365, 1825, np.inf]
timing_labels = ['Post-index/negative timing flag', 'Same day', '1-30 days before', '31-90 days before', '91-365 days before', '1-5 years before', '>5 years before']
q2_timing['CRP timing window'] = pd.cut(q2_timing['days_before_numeric'], bins=timing_bins, labels=timing_labels, right=True)

timing_rows = []
for label, sub in q2_timing.groupby('CRP timing window', observed=False):
    if len(sub) == 0:
        continue
    timing_rows.append({
        'CRP timing window': str(label),
        'N': fmt_int(len(sub)),
        'Percent of Q2 dataset': fmt_num(len(sub) / len(q2_timing) * 100, 1),
        'CRP, mg/L, median (IQR)': median_iqr(sub['crp_numeric'], 2),
        'CRP >10 mg/L, n (%)': n_pct(int((sub['crp_numeric'] > 10).sum()), len(sub)),
        'Ambiguous-unit flag, n (%)': n_pct(int((sub['ambiguous_numeric'] == 1).sum()), len(sub)) if 'ambiguous' in q2_timing.columns else '',
    })

crp_overall = pd.DataFrame([{
    'Metric': 'Q2 CRP dataset N',
    'Value': fmt_int(len(q2_timing)),
}, {
    'Metric': 'CRP, mg/L, median (IQR)',
    'Value': median_iqr(q2_timing['crp_numeric'], 2),
}, {
    'Metric': 'CRP, mg/L, range',
    'Value': range_cell(q2_timing['crp_numeric'], 2),
}, {
    'Metric': 'CRP >10 mg/L, n (%)',
    'Value': n_pct(int((q2_timing['crp_numeric'] > 10).sum()), len(q2_timing)),
}, {
    'Metric': 'Ambiguous-unit flag, n (%)',
    'Value': n_pct(int((q2_timing['ambiguous_numeric'] == 1).sum()), len(q2_timing)) if 'ambiguous' in q2_timing.columns else '',
}, {
    'Metric': 'Days before index/target, median (IQR)',
    'Value': median_iqr(q2_timing['days_before_numeric'], 1),
}, {
    'Metric': 'Days before index/target, range',
    'Value': range_cell(q2_timing['days_before_numeric'], 0),
}])
save_table('q2_supp_crp_overall_audit_c126_c129', crp_overall, 'Supplementary table. Q2 CRP overall audit')
save_table('q2_supp_crp_timing_distribution_c129', pd.DataFrame(timing_rows), 'Supplementary table. Q2 CRP timing distribution')

def fit_ols_formula(data, formula):
    if smf is None:
        raise RuntimeError('statsmodels is not available')
    fit = smf.ols(formula=formula, data=data).fit()
    ci = fit.conf_int()
    return fit, ci

def q2_window_sensitivity_rows():
    if 'days_before_numeric' not in q2_timing.columns:
        return pd.DataFrame([{'Window': 'Not generated', 'Model': '', 'N': '', 'Beta for log 25(OH)D': '', '95% CI': '', 'P value': '', 'R-squared': '', 'Reason': 'days_before column not available.'}])
    windows = [
        ('Unrestricted nearest pre-index', q2_timing['days_before_numeric'].notna()),
        ('CRP within 30 days before index/target', q2_timing['days_before_numeric'].between(0, 30, inclusive='both')),
        ('CRP within 90 days before index/target', q2_timing['days_before_numeric'].between(0, 90, inclusive='both')),
        ('CRP within 365 days before index/target', q2_timing['days_before_numeric'].between(0, 365, inclusive='both')),
    ]
    rows = []
    for window_label, mask in windows:
        sub = q2_timing.loc[mask].copy()
        if len(sub) == 0:
            rows.append({'Window': window_label, 'Model': 'All', 'N': '0', 'Beta for log 25(OH)D': '', '95% CI': '', 'P value': '', 'R-squared': '', 'Reason': 'No records in window.'})
            continue
        if stats is not None:
            corr_data = sub[['log_vitd', 'log_crp']].dropna()
            if len(corr_data) >= 3:
                pearson = stats.pearsonr(corr_data['log_vitd'], corr_data['log_crp'])
                spearman = stats.spearmanr(sub['vitd_main'], sub['crp_mgL'], nan_policy='omit')
                rows.append({'Window': window_label, 'Model': 'Correlation summary', 'N': fmt_int(len(corr_data)), 'Beta for log 25(OH)D': '', '95% CI': '', 'P value': '', 'R-squared': '', 'Pearson log-log r': fmt_num(pearson.statistic, 3), 'Spearman raw rho': fmt_num(spearman.statistic, 3), 'Reason': ''})
        formulas = [('Unadjusted', 'log_crp ~ log_vitd', ['log_crp', 'log_vitd'])]
        for _, spec in q2_models.iterrows():
            model_name = Q2_MODEL_LABELS.get(spec.get('model', ''), spec.get('model', ''))
            fields = [f.strip() for f in str(spec.get('complete_case_fields', '')).split(',') if f.strip() in sub.columns]
            formulas.append((model_name, spec.get('formula', ''), fields))
        for model_name, formula, fields in formulas:
            if not formula:
                continue
            data = sub.dropna(subset=fields).copy() if fields else sub.copy()
            try:
                fit, ci = fit_ols_formula(data, formula)
                beta = float(fit.params['log_vitd'])
                rows.append({
                    'Window': window_label,
                    'Model': model_name,
                    'N': fmt_int(fit.nobs),
                    'Beta for log 25(OH)D': fmt_num(beta, 3),
                    '95% CI': f"{float(ci.loc['log_vitd', 0]):.3f} to {float(ci.loc['log_vitd', 1]):.3f}",
                    'P value': fmt_p(fit.pvalues['log_vitd']),
                    'R-squared': fmt_num(fit.rsquared, 3),
                    'Pearson log-log r': '',
                    'Spearman raw rho': '',
                    'Reason': '',
                })
            except Exception as exc:
                rows.append({'Window': window_label, 'Model': model_name, 'N': fmt_int(len(data)), 'Beta for log 25(OH)D': '', '95% CI': '', 'P value': '', 'R-squared': '', 'Pearson log-log r': '', 'Spearman raw rho': '', 'Reason': str(exc)})
    return pd.DataFrame(rows)

q2_timing_sensitivity = q2_window_sensitivity_rows()
save_table('q2_supp_crp_30_90_timing_sensitivity_c129', q2_timing_sensitivity, 'Supplementary table. Q2 CRP timing-window sensitivity', png_preview=False)


## BMI and eGFR availability/timing


In [ ]:
# BMI range/timing and eGFR availability/timing audit

def bmi_range_rows(df, cohort_label, value_col='bmi_full', source_col=None):
    if value_col not in df.columns:
        return []
    temp = df.copy()
    temp[value_col] = pd.to_numeric(temp[value_col], errors='coerce')
    bands = [
        ('<12 kg/m2', temp[value_col] < 12),
        ('12 to <18.5 kg/m2', temp[value_col].between(12, 18.5, inclusive='left')),
        ('18.5 to <25 kg/m2', temp[value_col].between(18.5, 25, inclusive='left')),
        ('25 to <30 kg/m2', temp[value_col].between(25, 30, inclusive='left')),
        ('>=30 kg/m2', temp[value_col] >= 30),
        ('Missing', temp[value_col].isna()),
    ]
    rows = [{
        'Cohort': cohort_label,
        'BMI category/audit metric': 'BMI, kg/m2, median (IQR)',
        'N': fmt_int(temp[value_col].notna().sum()),
        'Percent': '',
        'BMI summary': median_iqr(temp[value_col], 1),
        'BMI range': range_cell(temp[value_col], 2),
        'Source detail': source_col or '',
    }]
    for label, mask in bands:
        rows.append({
            'Cohort': cohort_label,
            'BMI category/audit metric': label,
            'N': fmt_int(mask.sum()),
            'Percent': fmt_num(mask.mean() * 100, 1),
            'BMI summary': '',
            'BMI range': '',
            'Source detail': source_col or '',
        })
    if source_col and source_col in temp.columns:
        for source, sub in temp.groupby(source_col, dropna=False):
            rows.append({
                'Cohort': cohort_label,
                'BMI category/audit metric': f'BMI source: {source}',
                'N': fmt_int(len(sub)),
                'Percent': fmt_num(len(sub) / len(temp) * 100, 1),
                'BMI summary': median_iqr(sub[value_col], 1),
                'BMI range': range_cell(sub[value_col], 2),
                'Source detail': source_col,
            })
    return rows

bmi_rows = []
bmi_rows.extend(bmi_range_rows(q1_baseline, 'Q1 primary analytic cohort', 'bmi_full', 'bmi_source' if 'bmi_source' in q1_baseline.columns else None))
bmi_rows.extend(bmi_range_rows(q1_baseline[q1_baseline['person_id'].isin(q1_cox['person_id'])], 'Q1 Cox modeling cohort', 'bmi_full', 'bmi_source' if 'bmi_source' in q1_baseline.columns else None))
bmi_rows.extend(bmi_range_rows(q2_dataset, 'Q2 CRP model dataset', 'bmi_full', None))
save_table('q1_q2_supp_bmi_range_source_audit_c139', pd.DataFrame(bmi_rows), 'Supplementary table. BMI range and source audit')

bmi_timing_rows = []
q1_bmi_timing = read_csv_optional(Q1_FILES['q1_bmi_timing_audit'])
if q1_bmi_timing is not None:
    for _, r in q1_bmi_timing.iterrows():
        bmi_timing_rows.append({'Source': 'Q1 recorded BMI timing audit', 'Metric': r.get('metric', ''), 'Value': r.get('value', '')})

q1_recovered_bmi_timing = read_csv_optional(Q1_FILES['q1_recovered_bmi_timing_audit'])
if q1_recovered_bmi_timing is not None:
    col = 'bmi_meas_days_before_index' if 'bmi_meas_days_before_index' in q1_recovered_bmi_timing.columns else q1_recovered_bmi_timing.columns[-1]
    x = pd.to_numeric(q1_recovered_bmi_timing[col], errors='coerce')
    bmi_timing_rows.extend([
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Rows with recovered BMI timing', 'Value': fmt_int(x.notna().sum())},
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Post-index recovered BMI timing, n (%)', 'Value': n_pct(int((x < 0).sum()), int(x.notna().sum()))},
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Same-day recovered BMI timing, n (%)', 'Value': n_pct(int((x == 0).sum()), int(x.notna().sum()))},
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Pre-index recovered BMI within 365 days, n (%)', 'Value': n_pct(int(x.between(0, 365, inclusive='both').sum()), int(x.notna().sum()))},
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Recovered BMI timing days before index, median (IQR)', 'Value': median_iqr(x, 0)},
        {'Source': 'Q1 recovered BMI timing audit', 'Metric': 'Recovered BMI timing days before index, range', 'Value': range_cell(x, 0)},
    ])

if not bmi_timing_rows:
    bmi_timing_rows.append({'Source': 'BMI timing audit', 'Metric': 'Status', 'Value': 'No BMI timing audit CSV found under DATA_DIR.'})
save_table('q1_supp_bmi_timing_audit_c141', pd.DataFrame(bmi_timing_rows), 'Supplementary table. BMI timing audit')

def summarize_egfr_file(path, label):
    df = read_csv_optional(path)
    if df is None:
        return pd.DataFrame([{'Source': label, 'Status': 'No eGFR detail file found', 'N': '', 'eGFR <60 n (%)': '', 'eGFR median (IQR)': '', 'eGFR range': '', 'Timing summary': ''}])
    egfr_candidates = [c for c in df.columns if c.lower() in {'egfr', 'baseline_egfr', 'egfr_ckd_epi_2021', 'egfr_2021'} or 'egfr' in c.lower()]
    egfr_col = egfr_candidates[0] if egfr_candidates else None
    timing_candidates = [c for c in df.columns if 'day' in c.lower() or 'before' in c.lower() or 'timing' in c.lower()]
    timing_col = timing_candidates[0] if timing_candidates else None
    if egfr_col is None:
        return pd.DataFrame([{'Source': label, 'Status': 'eGFR file found but no eGFR-like column detected', 'N': fmt_int(len(df)), 'eGFR <60 n (%)': '', 'eGFR median (IQR)': '', 'eGFR range': '', 'Timing summary': f'Columns: {", ".join(df.columns)}'}])
    egfr = pd.to_numeric(df[egfr_col], errors='coerce')
    timing_summary = ''
    if timing_col is not None:
        timing = pd.to_numeric(df[timing_col], errors='coerce')
        timing_summary = f'{timing_col}: median (IQR) {median_iqr(timing, 0)}; range {range_cell(timing, 0)}'
    return pd.DataFrame([{
        'Source': label,
        'Status': f'Read {Path(path).name}; eGFR column {egfr_col}',
        'N': fmt_int(len(df)),
        'eGFR <60 n (%)': n_pct(int((egfr < 60).sum()), int(egfr.notna().sum())),
        'eGFR median (IQR)': median_iqr(egfr, 1),
        'eGFR range': range_cell(egfr, 1),
        'Timing summary': timing_summary,
    }])

egfr_table = pd.concat([
    summarize_egfr_file(Q1_FILES['q1_egfr_baseline'], 'Q1 eGFR baseline file'),
    summarize_egfr_file(Q2_FILES['q2_egfr_baseline'], 'Q2 eGFR baseline file'),
], ignore_index=True)
if q2_coverage is not None and 'covariate' in q2_coverage.columns:
    kidney_rows = q2_coverage[q2_coverage['covariate'].astype(str).str.contains('Kidney|eGFR', case=False, na=False)].copy()
    if not kidney_rows.empty:
        kidney_rows.insert(0, 'Source', 'Q2 covariate coverage file')
        save_table('q2_supp_egfr_coverage_c144', kidney_rows, 'Supplementary table. Q2 eGFR coverage status', png_preview=False)
save_table('q1_q2_supp_egfr_availability_timing_c144', egfr_table, 'Supplementary table. eGFR availability and timing audit')


## Baseline missingness


In [ ]:
VITD_LABELS = ['Very low (<10 ng/mL)', 'Low (10 to <20 ng/mL)', 'Intermediate (20 to <30 ng/mL)', 'Higher/reference (>=30 ng/mL)']

def add_vitd_category(df, value_col):
    out = df.copy()
    out['vitd_category'] = pd.cut(
        pd.to_numeric(out[value_col], errors='coerce'),
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=VITD_LABELS,
        right=False,
    )
    return out

q1_table1 = add_vitd_category(q1_baseline, 'vitd_ng_ml').merge(
    q1_followup[['person_id', 'follow_up_years', 'event_mi_or_stroke', 'first_mi_date', 'first_stroke_date', 'first_event_date']],
    on='person_id',
    how='left',
    validate='one_to_one',
)
missing_vars = [
    ('25(OH)D', 'vitd_ng_ml'),
    ('Age', 'age_at_index'),
    ('Gender', 'gender'),
    ('Race/ethnicity', 'race_eth_simplified'),
    ('BMI', 'bmi_full'),
    ('Baseline diabetes', 'baseline_diabetes'),
    ('Baseline hypertension', 'baseline_hypertension'),
    ('Baseline hyperlipidemia', 'baseline_hyperlipidemia'),
    ('Follow-up years', 'follow_up_years'),
    ('Incident MI or stroke', 'event_mi_or_stroke'),
]
miss_rows = []
for label, col in missing_vars:
    s = q1_table1[col]
    miss_rows.append({'Variable': label, 'Column': col, 'Nonmissing n': fmt_int(s.notna().sum()), 'Missing n': fmt_int(s.isna().sum()), 'Missing %': fmt_num(s.isna().mean() * 100, 1)})
q1_missing = pd.DataFrame(miss_rows)
save_table('q1_supp_table1_missingness_inventory', q1_missing, 'Supplementary table. Q1 Table 1 missingness inventory')


## Write quality summaries


In [ ]:
markdown_workbook_path = write_markdown_workbook()
excel_path = write_excel_workbook()
manifest = pd.DataFrame(MANIFEST)
manifest['markdown_workbook'] = str(markdown_workbook_path)
manifest['excel_workbook'] = str(excel_path) if excel_path is not None else ''
manifest.to_csv(OUT_DIR / 'manifest_q1_q2_table_outputs.csv', index=False, encoding='utf-8-sig')
manifest
